# Install and fetch the benchmark

In this notebook, we walk through installing the library, fetching the benchmark, and running one algorithm on one problem instance. We use the single-coefficient estimation (SCE) task throughout. The steps are:

1. Install the library, in Terminal.
2. Check that this notebook runs in that installation.
3. Fetch the public corpus.
4. Open the SCE collection.
5. Run one trial.

Run the cells in order, from top to bottom. The last cell prints the algorithm's output on each seed when everything works.

## Before you run this notebook

This notebook will help fetch the benchmark, but before that, we must install the library. Do the following steps:

1. Install Python 3.13 from [python.org](https://www.python.org/downloads/) if you do not have it. Check it with `python3.13 --version`.
2. Run the following commands in Terminal to create a folder with its own Python environment and install the library and JupyterLab into it:
```bash
mkdir dpstats-start && cd dpstats-start
python3.13 -m venv .venv
.venv/bin/python -m pip install "dpstats @ git+https://github.com/saminulh/dpstats.git@v0.1.0" jupyterlab
```
3. Download this notebook into that folder (the download link is at the top of this page), then open it in JupyterLab with
```bash
.venv/bin/python -m jupyterlab install_and_fetch.ipynb
```
4. In JupyterLab, choose the Python 3 kernel, and now you can proceed with running the cells in this notebook.

To use VS Code instead, open the `dpstats-start` folder, install the Python and Jupyter extensions, and open this notebook. Choose **Select Kernel**, then **Python Environments**, and pick the folder's `.venv/bin/python`.

When a new version of the library is released, run the same install command with the new tag in the same folder.

## Checking the installation

The next cell checks that the notebook runs in the environment you created and imports the library. It prints the Python and library versions. If the cell fails, choose the folder's `.venv` as the kernel and run it again.

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

from dpstats import fetch_benchmark, open_collection

if sys.version_info < (3, 10):
    raise RuntimeError("Choose the folder's .venv Python as this notebook's kernel.")
print("Python:", sys.version.split()[0])
print("dpstats:", version("dpstats"))

## Fetching the public corpus

The benchmark has a private corpus that is withheld for official ranking and a public corpus you may use for more fine-grained analysis.

The public corpus is hosted on Hugging Face. The next cell downloads it, and the downloaded files will appear in `dp-benchmark/official-release/`. That directory holds the SCE data and the other tasks used in later notebooks. If the data is already downloaded, the cell skips the download. When you create your own algorithms or run your own algorithm evaluations, the results will be stored in `dp-benchmark/local`.

In [ ]:
workspace = fetch_benchmark(Path("dp-benchmark"))

## Opening the SCE collection

Each task's instances and configured algorithms are kept in a collection. The next cell opens the [SCE](https://saminulh.github.io/dpstats/single_coefficient_estimation/) collection with [open_collection](https://saminulh.github.io/dpstats/library/reference/collections/dpstats.open_collection/). It prints the release version, then the number of instances and configured algorithms in the release.

In [ ]:
from dpstats.tasks import SingleCoefficientEstimation

collection = open_collection(workspace, SingleCoefficientEstimation)
release = collection.release
print(f"release {release.version}, manifest {release.manifest_id}")
instances = collection.load_instances(source="release")
algorithms = collection.load_algorithms(source="release")
print(f"SCE release: {len(instances)} instances, {len(algorithms)} algorithms")

## Running one trial

A trial runs one algorithm on one instance with one privacy budget and one seed. The next cell picks the smallest instance with public bounds on its data and runs [SSP](https://saminulh.github.io/dpstats/linear_regression/algorithms/SSP/) on it with $\varepsilon = 3$ and $\delta = 10^{-5}$ and three different seeds. SSP is a linear regression algorithm, so the [ToLinearRegression](https://saminulh.github.io/dpstats/single_coefficient_estimation/transformations/ToLinearRegression/) transformation turns the SCE instance into a linear regression one and returns the requested coefficient from the fit.

The cell prints the outputs on those seeds, showing that the download process worked.

In [ ]:
import numpy as np

from dpstats.algorithm import CompositeAlgorithm
from dpstats.budgets import ApproxDPBudget
from dpstats.tasks.linear_regression.algorithms import SSP
from dpstats.tasks.single_coefficient_estimation.transformations import ToLinearRegression


def is_bounded(instance):
    metadata = instance.metadata
    return np.isfinite(metadata.design_domain.l2_norm_bound) and np.isfinite(
        metadata.response_domain.l2_norm_bound
    )


instance = min(
    (item for item in instances.values() if is_bounded(item)),
    key=lambda item: item.metadata.n_samples,
)
algorithm = CompositeAlgorithm(transformations=(ToLinearRegression(),), algorithm=SSP())
results = collection.run_sweep(
    algorithms=[algorithm],
    instances=[instance],
    budgets=[ApproxDPBudget(eps=3.0, delta=1e-5)],
    seeds=[0, 1, 2],
    time_limit=None,
    memory_limit_gb=None,
)
for _, row in results.sort_values("alg_seed").iterrows():
    print(f"seed {row['alg_seed']} ({row['status'].value}): {row['payload'].coefficient:.4g}")

## Next steps

[Run your own script](run_your_own_script.ipynb) shows how to quickly implement within our library a method you already have and run a few trials. [Compare on the benchmark](compare_on_the_benchmark.ipynb) then reads your results beside the released ones.

To learn the library's concepts first, start with the [Introduction notebook](https://saminulh.github.io/dpstats/library/tutorials/tasks/).